# Modelo Random Forest

Clasificacion de precio anunciado por noche.


In [1]:
# Importamos utilidades de rutas y del sistema
from pathlib import Path
import sys

# Detectamos la raiz del repositorio para poder importar src/
ROOT = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent

# Comprobamos que estamos en la carpeta correcta
assert (ROOT / "src/preprocessing.py").is_file(), "Abre la raiz o notebooks/"

# Anadimos la raiz al path de Python
sys.path.insert(0, str(ROOT))

# Librerias estandar
import hashlib
import platform
import subprocess

# Librerias de datos
import numpy as np
import pandas as pd
from IPython.display import display

# Funciones de preparacion de datos
from src.preprocessing import load_train_test, RAW_DATA_PATH, TARGET_LABELS

# Funciones y constantes del modelo Random Forest
from src.rf_model import (
    SEED, N_TRIALS, GAP_LIMIT, SEARCH_SPACE, FIXED_PARAMS, CV_DESCRIPTION,
    LABEL_TO_INT, COMPARISON_COLUMNS, encode_target, make_folds, make_pipeline,
    cv_metrics, run_search, select_trial, oof_diagnostics, comparison_row, trials_table,
)


## 1. Carga comun


In [2]:
# Cargamos, limpiamos y dividimos los datos; el test se reserva sin inspeccionarlo
X_train, reserved_X, y_labels, reserved_y, thresholds = load_train_test(return_thresholds=True)

# Eliminamos los objetos reservados para no usar el test
del reserved_X, reserved_y

# Convertimos las etiquetas de texto a numeros (Economico=0, Medio=1, Premium=2)
y_train = encode_target(y_labels)

# Creamos las particiones de validacion cruzada
folds = make_folds(X_train, y_train)

# Guardamos metadatos de procedencia
provenance = {
    "snapshot": "2026-09-17",
    "raw_sha256": hashlib.sha256(RAW_DATA_PATH.read_bytes()).hexdigest(),
    "code_commit": subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip(),
    "python": platform.python_version(),
}

# Mostramos tamanos
print("X_train", X_train.shape, "y_train", y_train.shape)


X_train (10088, 23) y_train (10088,)


## 2. Referencias


In [3]:
# Pipeline con DummyClassifier (siempre predice la clase mas frecuente)
dummy_pipeline = make_pipeline(dummy=True)

# Pipeline con Random Forest usando parametros por defecto
default_pipeline = make_pipeline()

# Metricas de validacion cruzada para ambos
dummy_cv = cv_metrics(dummy_pipeline, X_train, y_train, folds)
default_cv = cv_metrics(default_pipeline, X_train, y_train, folds)

# Mostramos la tabla comparativa
display(pd.DataFrame([dummy_cv, default_cv], index=["Dummy", "Random Forest referencia"]).drop(columns="folds"))


,cv_train_f1_macro,cv_validation_f1_macro,cv_validation_std,gap,gap_under_005,cv_fit_seconds
Dummy,0.166791,0.166791,0.000068,8.618269e-09,True,0.238719
Random Forest referencia,0.989145,0.777889,0.008139,2.112552e-01,False,3.312597


## 3. Optuna


In [4]:
# Mostramos el espacio de busqueda
display(pd.DataFrame(SEARCH_SPACE).T)

# Ejecutamos la busqueda de hiperparametros (solo train)
study, search_seconds = run_search(X_train, y_train, folds, n_trials=N_TRIALS)

# Convertimos los resultados a tabla
trial_results = trials_table(study)

# Guardamos los resultados en CSV
RESULTS = ROOT / "docs/results"
RESULTS.mkdir(parents=True, exist_ok=True)
trial_results.to_csv(RESULTS / "rf_trials.csv", index=False)

# Seleccionamos la mejor prueba valida (gap < 0.05)
selected = select_trial(study)
selected_params = selected.params
selected_pipeline = make_pipeline(selected_params)

print("selected trial", selected.number, selected.value)


,kind,low,high,step
n_estimators,int,100,500,50
max_depth,int,3,20,NaN
min_samples_split,int,2,20,NaN
min_samples_leaf,int,1,10,NaN
max_features,float,0.3,1.0,NaN


[I 2026-11-16 20:04:56,832] A new study created in memory with name: no-name-493dcefd-b4ac-4036-8968-f4eda30661d5
[I 2026-11-16 20:05:12,478] Trial 0 finished with value: 0.7691771294139661 and parameters: {'n_estimators': 250, 'max_depth': 20, 'min_samples_split': 15, 'min_samples_leaf': 6, 'max_features': 0.40921304830970556}. Best is trial 0 with value: 0.7691771294139661.
[I 2026-11-16 20:05:19,939] Trial 1 finished with value: 0.7087266940458796 and parameters: {'n_estimators': 150, 'max_depth': 4, 'min_samples_split': 18, 'min_samples_leaf': 7, 'max_features': 0.7956508044572318}. Best is trial 0 with value: 0.7691771294139661.
[I 2026-11-16 20:05:26,967] Trial 2 finished with value: 0.7689273577526117 and parameters: {'n_estimators': 100, 'max_depth': 20, 'min_samples_split': 17, 'min_samples_leaf': 3, 'max_features': 0.42727747704497043}. Best is trial 0 with value: 0.7691771294139661.
[I 2026-11-16 20:05:35,513] Trial 3 finished with value: 0.752791458792852 and parameters: {'

selected trial 8 0.7593537479055246


## 4. OOF


In [5]:
# Calculamos predicciones fuera de fold para Dummy
dummy_oof = oof_diagnostics(dummy_pipeline, X_train, y_train, folds)

# Calculamos predicciones fuera de fold para el modelo seleccionado
selected_oof = oof_diagnostics(selected_pipeline, X_train, y_train, folds)

# Mostramos metricas OOF
for name, diagnostics in [("Dummy", dummy_oof), ("Random Forest seleccionado", selected_oof)]:
    print(name)
    display(pd.Series({key: diagnostics[key] for key in ["oof_accuracy", "oof_f1_macro", "oof_extreme_error_rate"]}))


Dummy


oof_accuracy              0.333664
oof_f1_macro              0.166791
oof_extreme_error_rate    0.333366
dtype: float64

Random Forest seleccionado


oof_accuracy              0.758822
oof_f1_macro              0.759376
oof_extreme_error_rate    0.009715
dtype: float64

## 5. Exportacion


In [6]:
# Construimos las filas de comparacion
rows = [
    comparison_row("DummyClassifier", dummy_cv, dummy_oof, params={"strategy": "most_frequent"},
                   search_seconds=0, provenance=provenance),
    comparison_row("RandomForest", cv_metrics(selected_pipeline, X_train, y_train, folds), selected_oof,
                   params={**FIXED_PARAMS, **selected_params}, search_seconds=search_seconds, provenance=provenance),
]

# Creamos el DataFrame final y lo guardamos
comparison = pd.DataFrame(rows, columns=COMPARISON_COLUMNS)
comparison.to_csv(RESULTS / "rf_comparison.csv", index=False)

# Mostramos las metricas principales
display(comparison[["model", "cv_validation_f1_macro", "gap", "oof_f1_macro"]])


,model,cv_validation_f1_macro,gap,oof_f1_macro
0,DummyClassifier,0.166791,8.618269e-09,0.166791
1,RandomForest,0.759354,4.612349e-02,0.759376
